In [1]:
# =========================
# Config + imports
# =========================
import json
import os
import time
import warnings
import tempfile
from pathlib import Path
from typing import Iterable, Mapping, Sequence

import arviz as az
import joblib
import numpy as np
import optuna
import pandas as pd
import pymc as pm

from IPython.display import display
from optuna.samplers import TPESampler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler


In [2]:
# =========================
# Setup and configure
# =========================
DATA_PATH = Path("../data/train_dataset_v3.csv")

FOLDER_PATH = Path("results/bayesian_hierarchical_regression_group_brand_slope_price")

TUNING_RESULTS_PATH = FOLDER_PATH / "3_tuned_results.csv"
BEST_CONFIG_PATH = FOLDER_PATH / "4_best_config.csv"
TEST_METRICS_PATH = FOLDER_PATH / "5_test_metrics.csv"
TEST_PREDICTIONS_PATH = FOLDER_PATH / "5_test_predictions.csv"
BEST_MODEL_PATH = FOLDER_PATH / "best_model.joblib"
BEST_TRACE_PATH = FOLDER_PATH / "best_trace.nc"

TARGET_COL = "nielsen_total_volume"
TIME_COL = "continuous_week"

CAT_COLS = [
    "product_sku_code",
    "customer",
    "top_brand",
    "flavor_internal",
    "pack_type_internal",
    "material_medium_description",
]

NUM_COLS = [
    "promotion_indicator",
    "pack_size_internal",
    "units_per_package_internal",
    "price_per_litre",
    "week_sin",
    "week_cos",
    "continuous_week",
]

SERIES_COLS = ["product_sku_code", "customer"]


TEST_WEEK_RATIO = 0.20
N_SPLITS = 3
RANDOM_STATE = 42
TUNE_TRIALS = 10
OPTUNA_SAMPLER_SEED = RANDOM_STATE
TUNING_OBJECTIVE = "mean_rmse"
TUNING_DIRECTION = "minimize"
MAX_FOLD_PREDICTION_MULTIPLIER = 20.0

MODEL_OBJECTIVE = "negative_binomial_log_link"
MONOTONE_CONSTRAINTS = {}

GROUP_KEY_COMPONENT_COLS = ["top_brand"]
USE_VARYING_INTERCEPT = True
VARYING_SLOPE_FEATURES = ["price_per_litre"]

GROUP_COL = "__series_group_key"    # Do not change this

FIXED_FEATURE_SET_NAME = "price+customer+promotion_indicator+top_brand+flavor_internal+pack_size_internal+units_per_package_internal+pack_type_internal+continuous_week+week_sin+week_cos"
FIXED_FEATURES = [
    "price_per_litre",
    "customer",
    "promotion_indicator",
    "top_brand",
    "flavor_internal",
    "pack_size_internal",
    "units_per_package_internal",
    "pack_type_internal",
    "week_sin",
    "week_cos",
    "continuous_week",
]

BAYESIAN_MODEL_CONFIG = {
    "model_objective": MODEL_OBJECTIVE,
    "group_col": GROUP_COL,
    "use_varying_intercept": USE_VARYING_INTERCEPT,
    "varying_slope_features": list(VARYING_SLOPE_FEATURES),
    "fixed_effect_prior_sd": 0.3,
    "global_intercept_prior_sd": 1.0,
    "group_intercept_prior_scale": 0.15,
    "group_slope_prior_scale": 0.15,
    "dispersion_prior_scale": 1.5,
}

TUNING_FIT_CONFIG = {
    "method": "advi",
    "advi_steps": 2000,
    "posterior_draws": 300,
    "random_seed": RANDOM_STATE,
}

FINAL_FIT_CONFIG = {
    "method": "nuts",
    "draws": 300,
    "tune": 300,
    "chains": 2,
    "cores": 1,
    "target_accept": 0.90,
    "max_treedepth": 12,
    "random_seed": RANDOM_STATE,
}

TUNED_SORT_COLS = [
    "tuned_mean_rmse",
    "tuned_std_rmse",
    "tuned_mean_rmsle",
    "feature_count",
]
TUNED_SORT_TOL = [0.01, None, None, None]

PREDICTION_QUANTILES = (0.05, 0.50, 0.95)
EPS = 1e-12
SEASONALITY = 1
LATENCY_REPEATS = 3

match MODEL_OBJECTIVE:
    case "negative_binomial_log_link" | "poisson_log_link":
        pass
    case _:
        raise ValueError(f"{MODEL_OBJECTIVE} is not a valid model objective.")

NUM_FEATURE_POOL = set(NUM_COLS)
CAT_FEATURE_POOL = set(CAT_COLS)


In [3]:
# =========================
# Validate fixed feature set
# =========================
feature_universe = NUM_FEATURE_POOL | CAT_FEATURE_POOL
unknown_fixed_features = sorted(set(FIXED_FEATURES) - feature_universe)
if unknown_fixed_features:
    raise ValueError(f"Fixed features are not in the numeric/categorical pools: {unknown_fixed_features}")
if GROUP_COL in FIXED_FEATURES:
    raise ValueError(f"GROUP_COL '{GROUP_COL}' must not appear in FIXED_FEATURES.")
missing_varying_slope_features = sorted(set(VARYING_SLOPE_FEATURES) - set(FIXED_FEATURES))
if missing_varying_slope_features:
    raise ValueError("All VARYING_SLOPE_FEATURES must be present in FIXED_FEATURES: " f"{missing_varying_slope_features}")
invalid_varying_slope_types = sorted(feature_name for feature_name in VARYING_SLOPE_FEATURES if feature_name not in NUM_FEATURE_POOL)
if invalid_varying_slope_types:
    raise ValueError("Varying-slope features must be numeric: " f"{invalid_varying_slope_types}")
print(f"Fixed feature set name: {FIXED_FEATURE_SET_NAME}")
print(f"Fixed features ({len(FIXED_FEATURES)}): {FIXED_FEATURES}")
print(f"Group column: {GROUP_COL}")
print(f"Group key components: {GROUP_KEY_COMPONENT_COLS}")
print(f"Use varying intercept: {USE_VARYING_INTERCEPT}")
print(f"Varying slope features: {VARYING_SLOPE_FEATURES}")

Fixed feature set name: price+customer+promotion_indicator+top_brand+flavor_internal+pack_size_internal+units_per_package_internal+pack_type_internal+continuous_week+week_sin+week_cos
Fixed features (11): ['price_per_litre', 'customer', 'promotion_indicator', 'top_brand', 'flavor_internal', 'pack_size_internal', 'units_per_package_internal', 'pack_type_internal', 'week_sin', 'week_cos', 'continuous_week']
Group column: __series_group_key
Group key components: ['top_brand']
Use varying intercept: True
Varying slope features: ['price_per_litre']


In [4]:
# =========================
# Helper methods
# =========================
def mae(y_true, y_pred):
    return float(mean_absolute_error(y_true, y_pred))


def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def r2(y_true, y_pred):
    return float(r2_score(y_true, y_pred))


def rmsle(y_true, y_pred):
    y_true_arr = np.asarray(y_true, dtype=float)
    y_pred_arr = np.asarray(y_pred, dtype=float)

    if np.any(y_true_arr < 0) or np.any(y_pred_arr < 0):
        warnings.warn(
            "RMSLE received negative values. They will be clipped to 0 before calculation.",
            RuntimeWarning,
        )
        y_true_arr = np.clip(y_true_arr, 0.0, None)
        y_pred_arr = np.clip(y_pred_arr, 0.0, None)

    return float(
        np.sqrt(np.mean((np.log1p(y_pred_arr) - np.log1p(y_true_arr)) ** 2))
    )


def mape_pct(y_true, y_pred, eps=EPS):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.maximum(np.abs(y_true), eps)
    return float(np.mean(np.abs(y_true - y_pred) / denom) * 100.0)


def smape_pct(y_true, y_pred, eps=EPS):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.maximum((np.abs(y_true) + np.abs(y_pred)) / 2.0, eps)
    return float(np.mean(np.abs(y_true - y_pred) / denom) * 100.0)


def wmape_pct(y_true, y_pred, eps=EPS):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.maximum(np.sum(np.abs(y_true)), eps)
    return float(np.sum(np.abs(y_true - y_pred)) / denom * 100.0)


def mean_bias(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.mean(y_pred - y_true))


def mean_pred_true_ratio(y_true, y_pred, eps=1.0):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.maximum(y_true, eps)
    return float(np.mean(y_pred / denom))


def pct_pred_gt_2x_true(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.mean(y_pred > (2.0 * y_true)) * 100.0)


def pct_pred_lt_half_true(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.mean(y_pred < (0.5 * y_true)) * 100.0)


def panel_mase(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    y_pred,
    series_cols,
    target_col,
    time_col,
    seasonality=1,
    eps=EPS,
):
    train_sorted = train_df.sort_values(series_cols + [time_col]).copy()

    scale_df = (
        train_sorted
        .groupby(series_cols, sort=False)[target_col]
        .apply(
            lambda s: np.mean(
                np.abs(s.to_numpy()[seasonality:] - s.to_numpy()[:-seasonality])
            )
            if len(s) > seasonality
            else np.nan
        )
        .reset_index(name="_scale")
    )

    eval_df = test_df[series_cols + [target_col]].copy()
    eval_df["_y_pred"] = np.asarray(y_pred, dtype=float)
    eval_df = eval_df.merge(scale_df, on=series_cols, how="left")

    valid = eval_df["_scale"].notna() & (eval_df["_scale"] > eps)
    if not valid.any():
        return float("nan")

    scaled_abs_err = (
        np.abs(eval_df.loc[valid, target_col] - eval_df.loc[valid, "_y_pred"])
        / eval_df.loc[valid, "_scale"]
    )
    return float(np.mean(scaled_abs_err))


In [5]:
def make_one_hot_encoder():
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            drop="first",
            sparse_output=False,
        )
    except TypeError:
        return OneHotEncoder(
            handle_unknown="ignore",
            drop="first",
            sparse=False,
        )


def safe_feature_token(feature_name: str) -> str:
    cleaned = "".join(ch if ch.isalnum() else "_" for ch in feature_name)
    return f"f_{cleaned}"


def flatten_posterior_draws(idata, var_name: str) -> np.ndarray:
    values = np.asarray(idata.posterior[var_name], dtype=float)
    return values.reshape(-1, *values.shape[2:])


def posterior_diagnostic_summary(idata) -> dict[str, float]:
    diagnostics = {
        "posterior_draws": np.nan,
        "divergences": np.nan,
        "max_rhat": np.nan,
        "min_ess_bulk": np.nan,
        "min_ess_tail": np.nan,
    }

    if "posterior" in idata.groups() and "alpha" in idata.posterior:
        alpha_values = np.asarray(idata.posterior["alpha"])
        diagnostics["posterior_draws"] = int(
            alpha_values.shape[0] * alpha_values.shape[1]
        )

    if "sample_stats" in idata.groups() and "diverging" in idata.sample_stats:
        diagnostics["divergences"] = int(
            np.asarray(idata.sample_stats["diverging"]).sum()
        )

    try:
        rhat_ds = az.rhat(idata, method="rank")
        rhat_values = np.asarray(rhat_ds.to_array(), dtype=float).ravel()
        rhat_values = rhat_values[np.isfinite(rhat_values)]
        if rhat_values.size:
            diagnostics["max_rhat"] = float(rhat_values.max())
    except Exception:
        pass

    try:
        ess_bulk_ds = az.ess(idata, method="bulk")
        ess_bulk_values = np.asarray(ess_bulk_ds.to_array(), dtype=float).ravel()
        ess_bulk_values = ess_bulk_values[np.isfinite(ess_bulk_values)]
        if ess_bulk_values.size:
            diagnostics["min_ess_bulk"] = float(ess_bulk_values.min())
    except Exception:
        pass

    try:
        ess_tail_ds = az.ess(idata, method="tail")
        ess_tail_values = np.asarray(ess_tail_ds.to_array(), dtype=float).ravel()
        ess_tail_values = ess_tail_values[np.isfinite(ess_tail_values)]
        if ess_tail_values.size:
            diagnostics["min_ess_tail"] = float(ess_tail_values.min())
    except Exception:
        pass

    return diagnostics


def validate_bayesian_feature_spec(
    *,
    feature_cols: Sequence[str],
    group_col: str,
    varying_slope_features: Sequence[str],
    num_feature_pool: Iterable[str],
    cat_feature_pool: Iterable[str],
) -> None:
    known_features = set(num_feature_pool) | set(cat_feature_pool) | {group_col}

    unknown_features = sorted(set(feature_cols) - known_features)
    if unknown_features:
        raise ValueError(f"Unknown feature columns: {unknown_features}")

    if group_col not in known_features:
        raise ValueError(f"GROUP_COL '{group_col}' is not in the feature pools.")

    if group_col in feature_cols:
        raise ValueError(
            f"GROUP_COL '{group_col}' must not appear in the fixed-effect feature list."
        )

    invalid_varying_slope_features = sorted(
        set(varying_slope_features) - set(feature_cols)
    )
    if invalid_varying_slope_features:
        raise ValueError(
            "All varying-slope features must be included in the active feature list: "
            f"{invalid_varying_slope_features}"
        )

    invalid_varying_slope_types = sorted(
        feature_name
        for feature_name in varying_slope_features
        if feature_name not in set(num_feature_pool)
    )
    if invalid_varying_slope_types:
        raise ValueError(
            "Varying-slope features must be numeric: "
            f"{invalid_varying_slope_types}"
        )


In [6]:
def build_design_state(
    *,
    train_df: pd.DataFrame,
    feature_cols: Sequence[str],
    target_col: str,
    group_col: str,
    num_feature_pool: Iterable[str],
    cat_feature_pool: Iterable[str],
    varying_slope_features: Sequence[str],
) -> dict[str, object]:
    validate_bayesian_feature_spec(
        feature_cols=feature_cols,
        group_col=group_col,
        varying_slope_features=varying_slope_features,
        num_feature_pool=num_feature_pool,
        cat_feature_pool=cat_feature_pool,
    )

    num_pool = set(num_feature_pool)
    cat_pool = set(cat_feature_pool)

    varying_slope_features = list(dict.fromkeys(varying_slope_features))
    fixed_effect_cols = [
        feature_name
        for feature_name in feature_cols
        if feature_name not in set(varying_slope_features)
    ]
    fixed_num_cols = [
        feature_name for feature_name in fixed_effect_cols if feature_name in num_pool
    ]
    fixed_cat_cols = [
        feature_name for feature_name in fixed_effect_cols if feature_name in cat_pool
    ]

    fixed_num_scaler = StandardScaler() if fixed_num_cols else None
    if fixed_num_scaler is not None:
        fixed_num_scaler.fit(train_df.loc[:, fixed_num_cols])

    varying_slope_scaler = StandardScaler() if varying_slope_features else None
    if varying_slope_scaler is not None:
        varying_slope_scaler.fit(train_df.loc[:, varying_slope_features])

    fixed_cat_encoder = make_one_hot_encoder() if fixed_cat_cols else None
    fixed_feature_names = list(fixed_num_cols)
    if fixed_cat_encoder is not None:
        fit_values = train_df.loc[:, fixed_cat_cols].astype("string").fillna("<NA>")
        fixed_cat_encoder.fit(fit_values)
        fixed_feature_names.extend(
            fixed_cat_encoder.get_feature_names_out(fixed_cat_cols).tolist()
        )

    group_values = train_df[group_col].astype("string").fillna("<NA>")
    group_levels = pd.Index(group_values.unique())
    group_to_index = {
        str(group_name): int(group_idx)
        for group_idx, group_name in enumerate(group_levels.tolist())
    }

    return {
        "target_col": target_col,
        "group_col": group_col,
        "feature_cols": list(feature_cols),
        "fixed_effect_cols": list(fixed_effect_cols),
        "fixed_num_cols": list(fixed_num_cols),
        "fixed_cat_cols": list(fixed_cat_cols),
        "fixed_feature_names": list(fixed_feature_names),
        "varying_slope_features": list(varying_slope_features),
        "varying_feature_tokens": {
            feature_name: safe_feature_token(feature_name)
            for feature_name in varying_slope_features
        },
        "fixed_num_scaler": fixed_num_scaler,
        "fixed_cat_encoder": fixed_cat_encoder,
        "varying_slope_scaler": varying_slope_scaler,
        "group_levels": group_levels.tolist(),
        "group_to_index": group_to_index,
    }


def transform_design(
    design_state: Mapping[str, object],
    df: pd.DataFrame,
    *,
    include_target: bool,
) -> dict[str, object]:
    n_rows = len(df)

    fixed_num_cols = list(design_state["fixed_num_cols"])
    fixed_cat_cols = list(design_state["fixed_cat_cols"])
    varying_slope_features = list(design_state["varying_slope_features"])

    matrices = []

    fixed_num_scaler = design_state["fixed_num_scaler"]
    if fixed_num_cols:
        fixed_num_matrix = fixed_num_scaler.transform(df.loc[:, fixed_num_cols])
        matrices.append(np.asarray(fixed_num_matrix, dtype=float))

    fixed_cat_encoder = design_state["fixed_cat_encoder"]
    if fixed_cat_cols:
        cat_values = df.loc[:, fixed_cat_cols].astype("string").fillna("<NA>")
        fixed_cat_matrix = fixed_cat_encoder.transform(cat_values)
        matrices.append(np.asarray(fixed_cat_matrix, dtype=float))

    if matrices:
        X_fixed = np.hstack(matrices).astype(float, copy=False)
    else:
        X_fixed = np.zeros((n_rows, 0), dtype=float)

    X_varying: dict[str, np.ndarray] = {}
    varying_slope_scaler = design_state["varying_slope_scaler"]
    if varying_slope_features:
        varying_matrix = varying_slope_scaler.transform(
            df.loc[:, varying_slope_features]
        )
        varying_matrix = np.asarray(varying_matrix, dtype=float)
        for feature_idx, feature_name in enumerate(varying_slope_features):
            X_varying[feature_name] = varying_matrix[:, feature_idx]

    group_col = str(design_state["group_col"])
    group_to_index = dict(design_state["group_to_index"])
    group_values = df[group_col].astype("string").fillna("<NA>")
    mapped_group_idx = group_values.map(group_to_index)
    group_idx_raw = mapped_group_idx.fillna(-1).astype(int).to_numpy()
    group_is_known = (group_idx_raw >= 0).astype(float)
    group_idx_safe = np.where(group_idx_raw >= 0, group_idx_raw, 0).astype("int64")

    design_bundle: dict[str, object] = {
        "n_rows": int(n_rows),
        "X_fixed": X_fixed,
        "X_varying": X_varying,
        "fixed_feature_names": list(design_state["fixed_feature_names"]),
        "group_idx_safe": group_idx_safe,
        "group_is_known": group_is_known,
        "group_levels": list(design_state["group_levels"]),
    }

    if include_target:
        target_values = pd.to_numeric(
            df[str(design_state["target_col"])],
            errors="raise",
        ).to_numpy(dtype=float)
        if np.any(target_values < 0):
            raise ValueError("Count likelihood requires a non-negative target.")
        if not np.allclose(target_values, np.round(target_values)):
            raise ValueError(
                "Count likelihood requires an integer-valued target."
            )
        design_bundle["y"] = np.round(target_values).astype("int64")

    return design_bundle


In [7]:
def posterior_prediction_summary(
    model_artifact: Mapping[str, object],
    score_df: pd.DataFrame,
    *,
    prediction_quantiles: Sequence[float] = PREDICTION_QUANTILES,
) -> dict[str, np.ndarray]:
    design_state = model_artifact["design_state"]
    idata = model_artifact["idata"]
    model_config = model_artifact["model_config"]
    design_bundle = transform_design(
        design_state,
        score_df,
        include_target=False,
    )

    n_rows = int(design_bundle["n_rows"])
    alpha_samples = flatten_posterior_draws(idata, "alpha").astype(float)
    eta_samples = np.broadcast_to(
        alpha_samples[:, None],
        (alpha_samples.shape[0], n_rows),
    ).copy()

    if design_bundle["X_fixed"].shape[1] > 0 and "beta_fixed" in idata.posterior:
        beta_fixed_samples = flatten_posterior_draws(idata, "beta_fixed").astype(float)
        eta_samples += beta_fixed_samples @ design_bundle["X_fixed"].T

    if bool(model_config["use_varying_intercept"]) and "group_intercept" in idata.posterior:
        group_intercept_samples = flatten_posterior_draws(
            idata,
            "group_intercept",
        ).astype(float)
        eta_samples += (
            group_intercept_samples[:, design_bundle["group_idx_safe"]]
            * design_bundle["group_is_known"][None, :]
        )

    varying_feature_tokens = dict(design_state["varying_feature_tokens"])
    for feature_name in design_state["varying_slope_features"]:
        feature_token = varying_feature_tokens[feature_name]
        feature_values = design_bundle["X_varying"][feature_name][None, :]

        global_slope_samples = flatten_posterior_draws(
            idata,
            f"{feature_token}_global_slope",
        ).astype(float)
        eta_samples += global_slope_samples[:, None] * feature_values

        group_slope_offset_samples = flatten_posterior_draws(
            idata,
            f"{feature_token}_group_slope_offset",
        ).astype(float)
        eta_samples += (
            group_slope_offset_samples[:, design_bundle["group_idx_safe"]]
            * design_bundle["group_is_known"][None, :]
            * feature_values
        )

    eta_samples = np.clip(eta_samples, -30.0, 30.0)
    mu_samples = np.exp(eta_samples)

    summary: dict[str, np.ndarray] = {
        "mean": mu_samples.mean(axis=0),
    }

    for q in prediction_quantiles:
        label = f"p{int(round(q * 100)):02d}"
        summary[label] = np.quantile(mu_samples, q, axis=0)

    return summary


def sort_with_tolerances(
    df: pd.DataFrame,
    sort_cols: Sequence[str],
    ascending: Sequence[bool] | None = None,
    tolerances: Sequence[float | None] | None = None,
) -> pd.DataFrame:
    if not sort_cols:
        raise ValueError("sort_cols must contain at least one column.")

    tolerances = [None] * len(sort_cols) if tolerances is None else list(tolerances)

    if len(sort_cols) != len(tolerances):
        raise ValueError("sort_cols and tolerances must have the same length.")

    if ascending is None:
        ascending = [True] * len(sort_cols)
    else:
        ascending = list(ascending)

    if len(sort_cols) != len(ascending):
        raise ValueError("sort_cols and ascending must have the same length.")

    if any(not isinstance(x, bool) for x in ascending):
        raise ValueError("ascending must contain only True/False values.")

    missing_cols = [col for col in sort_cols if col not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing sort columns: {missing_cols}")

    groups = [df.copy()]

    for col, tol, asc in zip(sort_cols, tolerances, ascending):
        next_groups = []

        for group in groups:
            if group.empty or len(group) == 1:
                next_groups.append(group)
                continue

            ordered = group.sort_values(by=col, ascending=asc, kind="stable")

            if tol is None:
                for _, subgroup in ordered.groupby(col, sort=False, dropna=False):
                    next_groups.append(subgroup.copy())
                continue

            if tol < 0:
                raise ValueError(f"Tolerance for '{col}' must be >= 0.")

            if not pd.api.types.is_numeric_dtype(ordered[col]):
                raise ValueError(
                    f"Column '{col}' is non-numeric, so its tolerance must be None."
                )

            remaining = ordered.copy()

            while not remaining.empty:
                best_value = float(remaining.iloc[0][col])
                delta = max(abs(best_value), 1e-12) * tol

                if asc:
                    mask = remaining[col] <= best_value + delta
                else:
                    mask = remaining[col] >= best_value - delta

                tier = remaining.loc[mask].copy()
                next_groups.append(tier)
                remaining = remaining.loc[~mask].copy()

        groups = next_groups

    sorted_df = pd.concat(groups, axis=0, ignore_index=True)
    return sorted_df.reset_index(drop=True)


def serialized_bayesian_size_bytes(
    artifact_bundle: Mapping[str, object],
    idata,
) -> int:
    tmp_joblib_path = None
    tmp_netcdf_path = None

    try:
        with tempfile.NamedTemporaryFile(suffix=".joblib", delete=False) as tmp_joblib:
            tmp_joblib_path = tmp_joblib.name

        with tempfile.NamedTemporaryFile(suffix=".nc", delete=False) as tmp_netcdf:
            tmp_netcdf_path = tmp_netcdf.name

        joblib.dump(dict(artifact_bundle), tmp_joblib_path)
        az.to_netcdf(idata, tmp_netcdf_path)

        return int(
            os.path.getsize(tmp_joblib_path)
            + os.path.getsize(tmp_netcdf_path)
        )
    finally:
        if tmp_joblib_path and os.path.exists(tmp_joblib_path):
            os.remove(tmp_joblib_path)
        if tmp_netcdf_path and os.path.exists(tmp_netcdf_path):
            os.remove(tmp_netcdf_path)


In [8]:
# =========================
# Load dataset
# =========================
FOLDER_PATH.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(DATA_PATH).copy()

missing_group_component_cols = sorted(set(GROUP_KEY_COMPONENT_COLS) - set(df.columns))
if missing_group_component_cols:
    raise ValueError(f"Missing group-key component columns: {missing_group_component_cols}")

if not GROUP_KEY_COMPONENT_COLS:
    raise ValueError("GROUP_KEY_COMPONENT_COLS must contain at least 1 column.")

df[GROUP_COL] = (
    df[GROUP_KEY_COMPONENT_COLS]
    .astype("string")
    .fillna("<NA>")
    .agg("||".join, axis=1)
)

print(f"Loaded rows: {len(df):,}")
print(f"Loaded columns: {len(df.columns):,}")
print(f"Derived group column: {GROUP_COL}")
DATA_PATH

Loaded rows: 92,777
Loaded columns: 21
Derived group column: __series_group_key


WindowsPath('../data/train_dataset_v3.csv')

In [9]:
# =========================
# Validate dataset
# =========================
required_cols = list(dict.fromkeys(CAT_COLS + NUM_COLS + SERIES_COLS + [TARGET_COL, TIME_COL, GROUP_COL] + FIXED_FEATURES))
missing_cols = sorted(set(required_cols) - set(df.columns))
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")
if df.empty:
    raise ValueError("The dataset is empty.")
_ = pd.to_numeric(df[TIME_COL], errors="raise")
target_numeric = pd.to_numeric(df[TARGET_COL], errors="raise").to_numpy(dtype=float)
if np.any(target_numeric < 0):
    raise ValueError("Count likelihood requires a non-negative target.")
if not np.allclose(target_numeric, np.round(target_numeric)):
    raise ValueError("Count likelihood requires an integer-valued target.")
df[TARGET_COL] = np.round(target_numeric).astype(int)
print(f"Rows: {len(df):,}")
print(f"Unique weeks: {df[TIME_COL].nunique():,}")
print(f"Unique groups: {df[GROUP_COL].nunique():,}")
print(f"Target: {TARGET_COL}")
print(f"Model objective: {MODEL_OBJECTIVE}")
print(f"Required columns validated: {len(required_cols)}")

Rows: 92,777
Unique weeks: 145
Unique groups: 18
Target: nielsen_total_volume
Model objective: negative_binomial_log_link
Required columns validated: 15


In [10]:
# =========================
# Time split train_df, test_df
# =========================
df = df.sort_values([TIME_COL] + SERIES_COLS).reset_index(drop=True).copy()
weeks = np.sort(df[TIME_COL].dropna().astype(int).unique())

if len(weeks) < 2:
    raise ValueError("Need at least 2 unique weeks for a train/test split.")

n_test_weeks = max(1, int(round(len(weeks) * TEST_WEEK_RATIO)))
n_test_weeks = min(n_test_weeks, len(weeks) - 1)
cut_week = int(weeks[-n_test_weeks])

train_df = df[df[TIME_COL] < cut_week].copy()
test_df = df[df[TIME_COL] >= cut_week].copy()

if train_df.empty or test_df.empty:
    raise ValueError("Train or test split is empty. Check TEST_WEEK_RATIO or data coverage.")

if int(train_df[TIME_COL].max()) >= int(test_df[TIME_COL].min()):
    raise ValueError("Time split is invalid: train overlaps test.")

print(f"Cut week: {cut_week}")
print(
    f"Train weeks: {int(train_df[TIME_COL].min())} -> {int(train_df[TIME_COL].max())} "
    f"({train_df[TIME_COL].nunique()} weeks)"
)
print(
    f"Test weeks:  {int(test_df[TIME_COL].min())} -> {int(test_df[TIME_COL].max())} "
    f"({test_df[TIME_COL].nunique()} weeks)"
)
print(f"Train rows: {len(train_df):,}")
print(f"Test rows:  {len(test_df):,}")


Cut week: 118
Train weeks: 0 -> 117 (116 weeks)
Test weeks:  118 -> 146 (29 weeks)
Train rows: 73,887
Test rows:  18,890


In [11]:
# =========================
# Define shared time-based CV on train_df
# =========================
# methods
def make_expanding_time_splits(
    df: pd.DataFrame,
    time_col: str,
    n_splits: int = 3,
) -> list[tuple[np.ndarray, np.ndarray]]:
    unique_periods = np.sort(df[time_col].dropna().astype(int).unique())

    if len(unique_periods) < n_splits + 1:
        raise ValueError(
            f"Not enough unique periods in '{time_col}' for {n_splits} expanding folds."
        )

    period_blocks = np.array_split(unique_periods, n_splits + 1)

    folds: list[tuple[np.ndarray, np.ndarray]] = []
    for fold_idx in range(1, n_splits + 1):
        train_periods = np.concatenate(period_blocks[:fold_idx])
        valid_periods = period_blocks[fold_idx]

        train_idx = np.flatnonzero(df[time_col].isin(train_periods).to_numpy())
        valid_idx = np.flatnonzero(df[time_col].isin(valid_periods).to_numpy())

        if len(train_idx) == 0 or len(valid_idx) == 0:
            raise ValueError(
                "Generated an empty train or validation fold. "
                "Check the time column coverage and n_splits."
            )

        folds.append((train_idx, valid_idx))

    return folds


def build_fold_summary(
    df: pd.DataFrame,
    time_col: str,
    cv_splits: Sequence[tuple[np.ndarray, np.ndarray]],
) -> pd.DataFrame:
    rows = []

    for fold_idx, (train_idx, valid_idx) in enumerate(cv_splits, start=1):
        train_periods = df.iloc[train_idx][time_col].astype(int)
        valid_periods = df.iloc[valid_idx][time_col].astype(int)

        rows.append(
            {
                "fold": fold_idx,
                "train_rows": int(len(train_idx)),
                "valid_rows": int(len(valid_idx)),
                "train_period_min": int(train_periods.min()),
                "train_period_max": int(train_periods.max()),
                "valid_period_min": int(valid_periods.min()),
                "valid_period_max": int(valid_periods.max()),
            }
        )

    return pd.DataFrame(rows)


shared_train_df = train_df.sort_values([TIME_COL] + SERIES_COLS).reset_index(drop=True).copy()
shared_cv_splits = make_expanding_time_splits(
    shared_train_df,
    time_col=TIME_COL,
    n_splits=N_SPLITS,
)
shared_fold_summary_df = build_fold_summary(
    shared_train_df,
    time_col=TIME_COL,
    cv_splits=shared_cv_splits,
)

print(f"Shared CV folds: {len(shared_cv_splits)}")
display(shared_fold_summary_df)


Shared CV folds: 3


,fold,train_rows,valid_rows,train_period_min,train_period_max,valid_period_min,valid_period_max
0,1,17836,18598,0,28,29,58
1,2,36434,18874,0,58,59,87
2,3,55308,18579,0,87,88,117


In [12]:
# =========================
# Stage 3 - define Bayesian fit methods
# =========================
# methods
def empty_posterior_diagnostics() -> dict[str, float]:
    return {"posterior_draws": np.nan, "divergences": np.nan, "max_rhat": np.nan, "min_ess_bulk": np.nan, "min_ess_tail": np.nan}


def build_bayesian_hierarchical_model(
    *,
    train_bundle: Mapping[str, object],
    model_config: Mapping[str, object],
):
    coords = {
        "group": list(train_bundle["group_levels"]),
    }
    if train_bundle["X_fixed"].shape[1] > 0:
        coords["fixed_feature"] = list(train_bundle["fixed_feature_names"])

    with pm.Model(coords=coords) as model:
        X_fixed = pm.Data(
            "X_fixed",
            np.asarray(train_bundle["X_fixed"], dtype=float),
        )
        group_idx = pm.Data(
            "group_idx",
            np.asarray(train_bundle["group_idx_safe"], dtype="int64"),
        )
        group_known = pm.Data(
            "group_known",
            np.asarray(train_bundle["group_is_known"], dtype=float),
        )

        alpha = pm.Normal(
            "alpha",
            mu=float(np.log(max(np.mean(train_bundle["y"]), 1e-6))),
            sigma=float(model_config["global_intercept_prior_sd"]),
        )
        eta = alpha

        if train_bundle["X_fixed"].shape[1] > 0:
            beta_fixed = pm.Normal(
                "beta_fixed",
                mu=0.0,
                sigma=float(model_config["fixed_effect_prior_sd"]),
                dims="fixed_feature",
            )
            eta = eta + pm.math.dot(X_fixed, beta_fixed)

        if bool(model_config["use_varying_intercept"]):
            sigma_group_intercept = pm.HalfNormal(
                "sigma_group_intercept",
                sigma=float(model_config["group_intercept_prior_scale"]),
            )
            z_group_intercept = pm.Normal(
                "z_group_intercept",
                mu=0.0,
                sigma=1.0,
                dims="group",
            )
            group_intercept = pm.Deterministic(
                "group_intercept",
                sigma_group_intercept * z_group_intercept,
                dims="group",
            )
            eta = eta + group_intercept[group_idx] * group_known

        varying_feature_tokens = dict(model_config["varying_feature_tokens"])
        for feature_name, feature_token in varying_feature_tokens.items():
            feature_values = pm.Data(
                f"{feature_token}_values",
                np.asarray(train_bundle["X_varying"][feature_name], dtype=float),
            )

            global_slope = pm.Normal(
                f"{feature_token}_global_slope",
                mu=0.0,
                sigma=float(model_config["fixed_effect_prior_sd"]),
            )

            sigma_group_slope = pm.HalfNormal(
                f"{feature_token}_sigma_group_slope",
                sigma=float(model_config["group_slope_prior_scale"]),
            )

            z_group_slope = pm.Normal(
                f"{feature_token}_z_group_slope",
                mu=0.0,
                sigma=1.0,
                dims="group",
            )

            group_slope_offset = pm.Deterministic(
                f"{feature_token}_group_slope_offset",
                sigma_group_slope * z_group_slope,
                dims="group",
            )

            eta = eta + global_slope * feature_values
            eta = eta + group_slope_offset[group_idx] * group_known * feature_values

        mu = pm.Deterministic("mu", pm.math.exp(eta))

        match model_config["model_objective"]:
            case "negative_binomial_log_link":
                nb_alpha = pm.HalfNormal(
                    "nb_alpha",
                    sigma=float(model_config["dispersion_prior_scale"]),
                )
                pm.NegativeBinomial(
                    "obs",
                    mu=mu,
                    alpha=nb_alpha,
                    observed=np.asarray(train_bundle["y"], dtype="int64"),
                )
            case "poisson_log_link":
                pm.Poisson(
                    "obs",
                    mu=mu,
                    observed=np.asarray(train_bundle["y"], dtype="int64"),
                )
            case _:
                raise ValueError(
                    f"Unsupported model objective: {model_config['model_objective']}"
                )

    return model



def fit_bayesian_hierarchical_model(
    *,
    train_df: pd.DataFrame,
    feature_cols: Sequence[str],
    target_col: str,
    group_col: str,
    num_feature_pool: Iterable[str],
    cat_feature_pool: Iterable[str],
    model_config: Mapping[str, object],
    fit_config: Mapping[str, object],
    include_diagnostics: bool = True,
) -> dict[str, object]:
    design_state = build_design_state(
        train_df=train_df,
        feature_cols=feature_cols,
        target_col=target_col,
        group_col=group_col,
        num_feature_pool=num_feature_pool,
        cat_feature_pool=cat_feature_pool,
        varying_slope_features=model_config["varying_slope_features"],
    )

    train_bundle = transform_design(design_state, train_df, include_target=True)
    runtime_model_config = {**dict(model_config), "varying_feature_tokens": dict(design_state["varying_feature_tokens"])}

    model = build_bayesian_hierarchical_model(train_bundle=train_bundle, model_config=runtime_model_config)

    t0 = time.perf_counter()
    with model:
        match fit_config["method"]:
            case "advi":
                approx = pm.fit(
                    n=int(fit_config["advi_steps"]),
                    method="advi",
                    random_seed=int(fit_config["random_seed"]),
                    progressbar=False,
                )
                idata = approx.sample(
                    draws=int(fit_config["posterior_draws"]),
                    random_seed=int(fit_config["random_seed"]),
                    return_inferencedata=True,
                )
            case "nuts":
                idata = pm.sample(
                    draws=int(fit_config["draws"]),
                    tune=int(fit_config["tune"]),
                    chains=int(fit_config["chains"]),
                    cores=int(fit_config["cores"]),
                    random_seed=int(fit_config["random_seed"]),
                    progressbar=False,
                    init="jitter+adapt_diag",
                    target_accept=float(fit_config["target_accept"]),
                    nuts_sampler="pymc",
                    nuts_sampler_kwargs={"max_treedepth": int(fit_config["max_treedepth"])},
                    return_inferencedata=True,
                )
            case _:
                raise ValueError(f"Unsupported fit method: {fit_config['method']}")
    fit_time_s = time.perf_counter() - t0

    diagnostics = posterior_diagnostic_summary(idata) if include_diagnostics else empty_posterior_diagnostics()

    return {
        "feature_cols": list(feature_cols),
        "group_col": group_col,
        "design_state": design_state,
        "model_config": runtime_model_config,
        "fit_config": dict(fit_config),
        "idata": idata,
        "fit_time_s": float(fit_time_s),
        "diagnostics": diagnostics,
    }

In [13]:
# =========================
# Stage 3 - run Optuna tuning on the fixed feature set
# =========================
# methods

def summarize_optional_diagnostics(values: Sequence[float], reducer: str) -> float:
    arr = np.asarray(values, dtype=float)
    finite = arr[np.isfinite(arr)]
    if finite.size == 0:
        return float("nan")
    match reducer:
        case "mean":
            return float(np.mean(finite))
        case "max":
            return float(np.max(finite))
        case "min":
            return float(np.min(finite))
        case _:
            raise ValueError(f"Unknown reducer: {reducer}")


def validate_fold_predictions(*, y_pred: np.ndarray, fold_train_df: pd.DataFrame, target_col: str, max_prediction_multiplier: float) -> None:
    if not np.all(np.isfinite(y_pred)):
        raise ValueError("Predictions contain non-finite values.")
    if y_pred.size == 0:
        raise ValueError("Predictions are empty.")
    train_target_max = float(fold_train_df[target_col].max())
    prediction_cap = max(train_target_max, 1.0) * float(max_prediction_multiplier)
    prediction_max = float(np.max(y_pred))
    if prediction_max > prediction_cap:
        raise ValueError("Predictions exceeded the stability cap: " f"max_pred={prediction_max:.6g}, cap={prediction_cap:.6g}.")


def validate_fold_metrics(metric_values: Mapping[str, float]) -> None:
    invalid_metrics = [metric_name for metric_name, metric_value in metric_values.items() if not np.isfinite(metric_value)]
    if invalid_metrics:
        raise ValueError(f"Metrics contain non-finite values: {invalid_metrics}")


def build_active_tuning_bounds(model_objective: str) -> dict[str, tuple[float, float]]:
    bounds: dict[str, tuple[float, float]] = {
        "fixed_effect_prior_sd": (0.1, 0.6),
        "global_intercept_prior_sd": (0.5, 2.0),
        "group_intercept_prior_scale": (0.05, 0.35),
    }
    if model_objective == "negative_binomial_log_link":
        bounds["dispersion_prior_scale"] = (0.8, 2.5)
    return bounds


def run_bayesian_cv(*, train_df: pd.DataFrame, target_col: str, feature_cols: Sequence[str], group_col: str, num_feature_pool: Iterable[str], cat_feature_pool: Iterable[str], cv_splits: Sequence[tuple[np.ndarray, np.ndarray]], model_config: Mapping[str, object], fit_config: Mapping[str, object], include_diagnostics: bool = True, max_prediction_multiplier: float = MAX_FOLD_PREDICTION_MULTIPLIER) -> dict[str, object]:
    fold_rmse: list[float] = []
    fold_rmsle: list[float] = []
    fold_r2: list[float] = []
    fold_mape: list[float] = []
    fold_smape: list[float] = []
    fold_wmape: list[float] = []
    fold_fit_time_s: list[float] = []
    fold_divergences: list[float] = []
    fold_max_rhat: list[float] = []
    fold_min_ess_bulk: list[float] = []

    for train_idx, valid_idx in cv_splits:
        fold_train_df = train_df.iloc[train_idx].reset_index(drop=True).copy()
        fold_valid_df = train_df.iloc[valid_idx].reset_index(drop=True).copy()
        artifact = fit_bayesian_hierarchical_model(train_df=fold_train_df, feature_cols=feature_cols, target_col=target_col, group_col=group_col, num_feature_pool=num_feature_pool, cat_feature_pool=cat_feature_pool, model_config=model_config, fit_config=fit_config, include_diagnostics=include_diagnostics)
        pred_summary = posterior_prediction_summary(artifact, fold_valid_df, prediction_quantiles=PREDICTION_QUANTILES)
        y_pred = np.clip(pred_summary["p50"], 0.0, None)
        y_true = fold_valid_df[target_col].to_numpy(dtype=float)
        validate_fold_predictions(y_pred=y_pred, fold_train_df=fold_train_df, target_col=target_col, max_prediction_multiplier=max_prediction_multiplier)
        metric_values = {"rmse": rmse(y_true, y_pred), "rmsle": rmsle(y_true, y_pred), "r2": r2(y_true, y_pred), "mape": mape_pct(y_true, y_pred), "smape": smape_pct(y_true, y_pred), "wmape": wmape_pct(y_true, y_pred)}
        validate_fold_metrics(metric_values)
        fold_rmse.append(metric_values["rmse"])
        fold_rmsle.append(metric_values["rmsle"])
        fold_r2.append(metric_values["r2"])
        fold_mape.append(metric_values["mape"])
        fold_smape.append(metric_values["smape"])
        fold_wmape.append(metric_values["wmape"])
        fold_fit_time_s.append(float(artifact["fit_time_s"]))
        fold_divergences.append(float(artifact["diagnostics"]["divergences"]))
        fold_max_rhat.append(float(artifact["diagnostics"]["max_rhat"]))
        fold_min_ess_bulk.append(float(artifact["diagnostics"]["min_ess_bulk"]))

    return {
        "fold_rmse": fold_rmse,
        "fold_rmsle": fold_rmsle,
        "fold_r2": fold_r2,
        "fold_mape": fold_mape,
        "fold_smape": fold_smape,
        "fold_wmape": fold_wmape,
        "fold_fit_time_s": fold_fit_time_s,
        "fold_divergences": fold_divergences,
        "fold_max_rhat": fold_max_rhat,
        "fold_min_ess_bulk": fold_min_ess_bulk,
        "mean_rmse": float(np.mean(fold_rmse)),
        "std_rmse": float(np.std(fold_rmse, ddof=0)),
        "mean_rmsle": float(np.mean(fold_rmsle)),
        "std_rmsle": float(np.std(fold_rmsle, ddof=0)),
        "mean_r2": float(np.mean(fold_r2)),
        "std_r2": float(np.std(fold_r2, ddof=0)),
        "mean_mape": float(np.mean(fold_mape)),
        "std_mape": float(np.std(fold_mape, ddof=0)),
        "mean_smape": float(np.mean(fold_smape)),
        "std_smape": float(np.std(fold_smape, ddof=0)),
        "mean_wmape": float(np.mean(fold_wmape)),
        "std_wmape": float(np.std(fold_wmape, ddof=0)),
        "mean_fit_time_s": float(np.mean(fold_fit_time_s)),
        "std_fit_time_s": float(np.std(fold_fit_time_s, ddof=0)),
        "mean_divergences": summarize_optional_diagnostics(fold_divergences, "mean"),
        "max_rhat": summarize_optional_diagnostics(fold_max_rhat, "max"),
        "min_ess_bulk": summarize_optional_diagnostics(fold_min_ess_bulk, "min"),
    }


def build_default_tuning_params(base_model_config: Mapping[str, object]) -> dict[str, float]:
    active_bounds = build_active_tuning_bounds(str(base_model_config["model_objective"]))
    return {param_name: float(base_model_config[param_name]) for param_name in active_bounds}


def build_tuning_objective(*, train_df: pd.DataFrame, target_col: str, feature_cols: Sequence[str], group_col: str, num_feature_pool: Iterable[str], cat_feature_pool: Iterable[str], cv_splits: Sequence[tuple[np.ndarray, np.ndarray]], base_model_config: Mapping[str, object], fit_config: Mapping[str, object], tuning_objective: str):
    active_bounds = build_active_tuning_bounds(str(base_model_config["model_objective"]))
    def objective(trial):
        try:
            tuned_params = {param_name: trial.suggest_float(param_name, low, high, log=True) for param_name, (low, high) in active_bounds.items()}
            model_config = {**dict(base_model_config), **tuned_params}
            cv_results = run_bayesian_cv(train_df=train_df, target_col=target_col, feature_cols=feature_cols, group_col=group_col, num_feature_pool=num_feature_pool, cat_feature_pool=cat_feature_pool, cv_splits=cv_splits, model_config=model_config, fit_config=fit_config, include_diagnostics=False)
            for metric_name in ["mean_rmse", "std_rmse", "mean_rmsle", "std_rmsle", "mean_r2", "std_r2", "mean_mape", "std_mape", "mean_smape", "std_smape", "mean_wmape", "std_wmape", "mean_fit_time_s", "std_fit_time_s"]:
                trial.set_user_attr(metric_name, float(cv_results[metric_name]))
            for metric_name in ["rmse", "rmsle", "r2", "mape", "smape", "wmape"]:
                for fold_idx, value in enumerate(cv_results[f"fold_{metric_name}"], start=1):
                    trial.set_user_attr(f"fold_{fold_idx}_{metric_name}", float(value))
            return float(cv_results[tuning_objective])
        except optuna.TrialPruned:
            raise
        except Exception as exc:
            trial.set_user_attr("pruned_reason", str(exc))
            raise optuna.TrialPruned(str(exc)) from exc
    return objective


def evaluate_fixed_feature_set_for_tuning(*, combination_id: int, groups: str, feature_cols: Sequence[str], train_df: pd.DataFrame, target_col: str, group_col: str, num_feature_pool: Iterable[str], cat_feature_pool: Iterable[str], cv_splits: Sequence[tuple[np.ndarray, np.ndarray]], base_model_config: Mapping[str, object], fit_config: Mapping[str, object], tuning_objective: str, tuning_direction: str, sampler_seed: int, tune_trials: int) -> dict[str, object]:
    study = optuna.create_study(direction=tuning_direction, sampler=TPESampler(seed=sampler_seed))
    study.enqueue_trial(build_default_tuning_params(base_model_config))
    t0 = time.perf_counter()
    study.optimize(build_tuning_objective(train_df=train_df, target_col=target_col, feature_cols=feature_cols, group_col=group_col, num_feature_pool=num_feature_pool, cat_feature_pool=cat_feature_pool, cv_splits=cv_splits, base_model_config=base_model_config, fit_config=fit_config, tuning_objective=tuning_objective), n_trials=tune_trials, show_progress_bar=False)
    tuning_time_s = time.perf_counter() - t0
    complete_trials = [trial for trial in study.trials if trial.state == optuna.trial.TrialState.COMPLETE]
    pruned_trials = [trial for trial in study.trials if trial.state == optuna.trial.TrialState.PRUNED]
    if not complete_trials:
        raise ValueError("No Optuna trials completed successfully.")
    best_trial = study.best_trial
    best_params = {**dict(base_model_config), **best_trial.params}
    row = {"combination_id": combination_id, "groups": groups, "feature_list_json": json.dumps(list(feature_cols), ensure_ascii=True), "feature_count": len(feature_cols), "tuning_trials": int(tune_trials), "n_successful_trials": int(len(complete_trials)), "n_pruned_trials": int(len(pruned_trials)), "tuning_time_s": float(tuning_time_s), "best_params_json": json.dumps(best_params, ensure_ascii=True)}
    for metric_name in ["mean_rmse", "std_rmse", "mean_rmsle", "std_rmsle", "mean_r2", "std_r2", "mean_mape", "std_mape", "mean_smape", "std_smape", "mean_wmape", "std_wmape", "mean_fit_time_s", "std_fit_time_s"]:
        row[f"tuned_{metric_name}"] = float(best_trial.user_attrs[metric_name])
    for fold_idx in range(1, len(cv_splits) + 1):
        for metric_name in ["rmse", "rmsle", "r2", "mape", "smape", "wmape"]:
            row[f"tuned_fold_{fold_idx}_{metric_name}"] = float(best_trial.user_attrs[f"fold_{fold_idx}_{metric_name}"])
    return row


def run_bayesian_tuning(*, train_df: pd.DataFrame, target_col: str, feature_cols: Sequence[str], group_label: str, group_col: str, num_feature_pool: Iterable[str], cat_feature_pool: Iterable[str], cv_splits: Sequence[tuple[np.ndarray, np.ndarray]], base_model_config: Mapping[str, object], fit_config: Mapping[str, object], tuning_objective: str, tuning_direction: str, sampler_seed: int, tune_trials: int) -> pd.DataFrame:
    return pd.DataFrame([evaluate_fixed_feature_set_for_tuning(combination_id=1, groups=group_label, feature_cols=feature_cols, train_df=train_df, target_col=target_col, group_col=group_col, num_feature_pool=num_feature_pool, cat_feature_pool=cat_feature_pool, cv_splits=cv_splits, base_model_config=base_model_config, fit_config=fit_config, tuning_objective=tuning_objective, tuning_direction=tuning_direction, sampler_seed=sampler_seed, tune_trials=tune_trials)]).reset_index(drop=True)

In [14]:
# =========================
# Stage 3 - execute Optuna tuning
# =========================
for stale_output_path in [
    TUNING_RESULTS_PATH,
    BEST_CONFIG_PATH,
    TEST_METRICS_PATH,
    TEST_PREDICTIONS_PATH,
    BEST_MODEL_PATH,
    BEST_TRACE_PATH,
]:
    if stale_output_path.exists():
        stale_output_path.unlink()
        print(f"Removed stale output: {stale_output_path}")

tuned_results_df = run_bayesian_tuning(
    train_df=shared_train_df,
    target_col=TARGET_COL,
    feature_cols=FIXED_FEATURES,
    group_label=FIXED_FEATURE_SET_NAME,
    group_col=GROUP_COL,
    num_feature_pool=NUM_FEATURE_POOL,
    cat_feature_pool=CAT_FEATURE_POOL,
    cv_splits=shared_cv_splits,
    base_model_config=BAYESIAN_MODEL_CONFIG,
    fit_config=TUNING_FIT_CONFIG,
    tuning_objective=TUNING_OBJECTIVE,
    tuning_direction=TUNING_DIRECTION,
    sampler_seed=OPTUNA_SAMPLER_SEED,
    tune_trials=TUNE_TRIALS,
)
print(f"Tuned combinations: {len(tuned_results_df)}")
display(tuned_results_df)

[I 2026-04-08 03:53:56,098] A new study created in memory with name: no-name-1f59c2bb-0b9f-4ebe-a921-1baa1866f8b8
c:\Users\Samuel\Desktop\fypcleanded\.venv\Lib\site-packages\pytensor\link\c\cmodule.py:2986: UserWarning: PyTensor could not link to a BLAS installation. Operations that might benefit from BLAS will be severely degraded.
This usually happens when PyTensor is installed via pip. We recommend it be installed via conda/mamba/pixi instead.
Alternatively, you can use an experimental backend such as Numba or JAX that perform their own BLAS optimizations, by setting `pytensor.config.mode == 'NUMBA'` or passing `mode='NUMBA'` when compiling a PyTensor function.
For more options and details see https://pytensor.readthedocs.io/en/latest/troubleshooting.html#how-do-i-configure-test-my-blas-library
  warnings.warn(
Finished [100%]: Average Loss = 2.7151e+05
c:\Users\Samuel\Desktop\fypcleanded\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown cate

Tuned combinations: 1


,combination_id,groups,feature_list_json,feature_count,tuning_trials,n_successful_trials,n_pruned_trials,tuning_time_s,best_params_json,tuned_mean_rmse,...,tuned_fold_2_r2,tuned_fold_2_mape,tuned_fold_2_smape,tuned_fold_2_wmape,tuned_fold_3_rmse,tuned_fold_3_rmsle,tuned_fold_3_r2,tuned_fold_3_mape,tuned_fold_3_smape,tuned_fold_3_wmape
0,1,price+customer+promotion_indicator+top_brand+f...,"[""price_per_litre"", ""customer"", ""promotion_ind...",11,10,10,0,1653.279547,"{""model_objective"": ""negative_binomial_log_lin...",23077.881986,...,-0.442021,95778.420765,143.370659,233.103702,21037.515163,3.126724,-0.317134,47551.192846,141.728104,224.985144


In [15]:
# =========================
# Stage 3 - save tuned results
# =========================
tuned_results_df.to_csv(TUNING_RESULTS_PATH, index=False)
print(f"Saved: {TUNING_RESULTS_PATH}")


Saved: results\bayesian_hierarchical_regression_group_brand_slope_price\3_tuned_results.csv


In [16]:
# =========================
# Stage 4 - load tuned results, rank, select top 1
# =========================s
tuned_results_df = pd.read_csv(TUNING_RESULTS_PATH)
if tuned_results_df.empty:
    raise ValueError("tuned_results.csv is empty. Cannot rank final candidates.")
sorted_tuned_results_df = sort_with_tolerances(df=tuned_results_df, sort_cols=TUNED_SORT_COLS, tolerances=TUNED_SORT_TOL).reset_index(drop=True)
display(sorted_tuned_results_df)
top1_df = sorted_tuned_results_df.head(1).reset_index(drop=True)
print(f"selected top {1} row out of {len(sorted_tuned_results_df)}")
display(top1_df)


,combination_id,groups,feature_list_json,feature_count,tuning_trials,n_successful_trials,n_pruned_trials,tuning_time_s,best_params_json,tuned_mean_rmse,...,tuned_fold_2_r2,tuned_fold_2_mape,tuned_fold_2_smape,tuned_fold_2_wmape,tuned_fold_3_rmse,tuned_fold_3_rmsle,tuned_fold_3_r2,tuned_fold_3_mape,tuned_fold_3_smape,tuned_fold_3_wmape
0,1,price+customer+promotion_indicator+top_brand+f...,"[""price_per_litre"", ""customer"", ""promotion_ind...",11,10,10,0,1653.279547,"{""model_objective"": ""negative_binomial_log_lin...",23077.881986,...,-0.442021,95778.420765,143.370659,233.103702,21037.515163,3.126724,-0.317134,47551.192846,141.728104,224.985144


selected top 1 row out of 1


,combination_id,groups,feature_list_json,feature_count,tuning_trials,n_successful_trials,n_pruned_trials,tuning_time_s,best_params_json,tuned_mean_rmse,...,tuned_fold_2_r2,tuned_fold_2_mape,tuned_fold_2_smape,tuned_fold_2_wmape,tuned_fold_3_rmse,tuned_fold_3_rmsle,tuned_fold_3_r2,tuned_fold_3_mape,tuned_fold_3_smape,tuned_fold_3_wmape
0,1,price+customer+promotion_indicator+top_brand+f...,"[""price_per_litre"", ""customer"", ""promotion_ind...",11,10,10,0,1653.279547,"{""model_objective"": ""negative_binomial_log_lin...",23077.881986,...,-0.442021,95778.420765,143.370659,233.103702,21037.515163,3.126724,-0.317134,47551.192846,141.728104,224.985144


In [17]:
# =========================
# Stage 4 - save best config
# =========================
top1_df.to_csv(BEST_CONFIG_PATH, index=False)
print(f"Saved: {BEST_CONFIG_PATH}")


Saved: results\bayesian_hierarchical_regression_group_brand_slope_price\4_best_config.csv


In [18]:
# =========================
# Stage 5 - refit best on full train and evaluate test
# =========================
def callable_latency_ms_per_row(predict_fn, X_like, n_repeats=LATENCY_REPEATS):
    _ = predict_fn(X_like.iloc[: min(len(X_like), 256)].copy())
    t0 = time.perf_counter()
    for _ in range(n_repeats):
        _ = predict_fn(X_like)
    elapsed = time.perf_counter() - t0
    return float((elapsed / n_repeats) * 1000.0 / max(len(X_like), 1))


def evaluate_predictions(*, model_name: str, tuning_method: str, refit_method: str, model_objective: str, tuning_objective: str, column_list: Sequence[str], y_true: Sequence[float] | np.ndarray, y_pred: Sequence[float] | np.ndarray, train_df: pd.DataFrame, test_df: pd.DataFrame, series_cols: Sequence[str], target_col: str, time_col: str, tuning_time_s: float = 0.0, refit_time_s: float = 0.0, end_to_end_time_s: float = 0.0, inference_latency_ms_per_row: float = np.nan, model_size_bytes: float = np.nan, model_params: Mapping[str, object] | None = None, monotone_constraints: Mapping[str, int] | None = None, posterior_diagnostics: Mapping[str, object] | None = None):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.clip(np.asarray(y_pred, dtype=float), 0.0, None)
    model_params = {} if model_params is None else dict(model_params)
    posterior_diagnostics = {} if posterior_diagnostics is None else dict(posterior_diagnostics)
    return {"Model": model_name, "Model Objective": model_objective, "Tuning Method": tuning_method, "Tuning Objective": tuning_objective, "Refit Method": refit_method, "column_list": json.dumps(list(column_list), ensure_ascii=True), "MAE": mae(y_true, y_pred), "RMSE": rmse(y_true, y_pred), "RMSLE": rmsle(y_true, y_pred), "R2": r2(y_true, y_pred), "MAPE": mape_pct(y_true, y_pred), "sMAPE": smape_pct(y_true, y_pred), "wMAPE": wmape_pct(y_true, y_pred), "MASE": panel_mase(train_df=train_df, test_df=test_df, y_pred=y_pred, series_cols=series_cols, target_col=target_col, time_col=time_col, seasonality=SEASONALITY), "Mean Bias": mean_bias(y_true, y_pred), "Mean Pred/True Ratio": mean_pred_true_ratio(y_true, y_pred), "Pct Pred > 2x True": pct_pred_gt_2x_true(y_true, y_pred), "Pct Pred < 0.5x True": pct_pred_lt_half_true(y_true, y_pred), "Tuning Time (s)": float(tuning_time_s), "Refit Time (s)": float(refit_time_s), "End-to-End Time (s)": float(end_to_end_time_s), "Inference Latency (ms/row)": float(inference_latency_ms_per_row), "Model Size (bytes)": float(model_size_bytes), "Posterior Draws": posterior_diagnostics.get("posterior_draws", np.nan), "Divergences": posterior_diagnostics.get("divergences", np.nan), "Max Rhat": posterior_diagnostics.get("max_rhat", np.nan), "Min ESS Bulk": posterior_diagnostics.get("min_ess_bulk", np.nan), "Model Params (json)": json.dumps(model_params, ensure_ascii=True), "Monotone Constraints Config (json)": json.dumps(monotone_constraints, ensure_ascii=True)}

top1_df = pd.read_csv(BEST_CONFIG_PATH)
if top1_df.empty:
    raise ValueError("best_config.csv is empty. Cannot run final test evaluation.")
if len(top1_df) != 1:
    raise ValueError(f"best_config.csv must contain exactly 1 row, got {len(top1_df)}.")
best_row = top1_df.iloc[0]
feature_cols = json.loads(best_row["feature_list_json"])
best_params = json.loads(best_row["best_params_json"])
tuning_time_s = float(best_row["tuning_time_s"])
missing_train_features = [col for col in feature_cols if col not in shared_train_df.columns]
missing_test_features = [col for col in feature_cols if col not in test_df.columns]
if missing_train_features:
    raise ValueError(f"Selected features are missing from shared_train_df: {missing_train_features}")
if missing_test_features:
    raise ValueError(f"Selected features are missing from test_df: {missing_test_features}")
final_artifact = fit_bayesian_hierarchical_model(train_df=shared_train_df, feature_cols=feature_cols, target_col=TARGET_COL, group_col=GROUP_COL, num_feature_pool=NUM_FEATURE_POOL, cat_feature_pool=CAT_FEATURE_POOL, model_config=best_params, fit_config=FINAL_FIT_CONFIG, include_diagnostics=True)
refit_time_s = float(final_artifact["fit_time_s"])
end_to_end_time_s = tuning_time_s + refit_time_s
prediction_summary = posterior_prediction_summary(final_artifact, test_df, prediction_quantiles=PREDICTION_QUANTILES)
y_test = test_df[TARGET_COL].to_numpy(dtype=float)
y_pred = np.clip(prediction_summary["p50"], 0.0, None)
validate_fold_predictions(y_pred=y_pred, fold_train_df=shared_train_df, target_col=TARGET_COL, max_prediction_multiplier=MAX_FOLD_PREDICTION_MULTIPLIER)
predict_fn = lambda df_like: np.clip(posterior_prediction_summary(final_artifact, df_like, prediction_quantiles=PREDICTION_QUANTILES)["p50"], 0.0, None)
inference_latency_ms_per_row = callable_latency_ms_per_row(predict_fn, test_df)
artifact_bundle = {"feature_cols": list(feature_cols), "group_col": GROUP_COL, "design_state": final_artifact["design_state"], "model_config": final_artifact["model_config"], "fit_config": final_artifact["fit_config"], "posterior_diagnostics": final_artifact["diagnostics"], "prediction_quantiles": list(PREDICTION_QUANTILES)}
model_size_bytes = serialized_bayesian_size_bytes(artifact_bundle, final_artifact["idata"])
final_model_params = {"model_config": best_params, "fit_config": dict(FINAL_FIT_CONFIG)}
metrics_row = evaluate_predictions(model_name="Bayesian Hierarchical Regression", tuning_method="Optuna TPE", refit_method=str(FINAL_FIT_CONFIG["method"]).upper(), model_objective=MODEL_OBJECTIVE, tuning_objective=TUNING_OBJECTIVE, column_list=feature_cols, y_true=y_test, y_pred=y_pred, train_df=shared_train_df, test_df=test_df, series_cols=SERIES_COLS, target_col=TARGET_COL, time_col=TIME_COL, tuning_time_s=tuning_time_s, refit_time_s=refit_time_s, end_to_end_time_s=end_to_end_time_s, inference_latency_ms_per_row=inference_latency_ms_per_row, model_size_bytes=model_size_bytes, model_params=final_model_params, monotone_constraints=MONOTONE_CONSTRAINTS, posterior_diagnostics=final_artifact["diagnostics"])
final_metrics_df = pd.DataFrame([metrics_row])
final_predictions_df = test_df.copy()
final_predictions_df["y_true"] = y_test
final_predictions_df["y_pred"] = y_pred
final_predictions_df["y_pred_p05"] = np.clip(prediction_summary["p05"], 0.0, None)
final_predictions_df["y_pred_p50"] = np.clip(prediction_summary["p50"], 0.0, None)
final_predictions_df["y_pred_p95"] = np.clip(prediction_summary["p95"], 0.0, None)
display(final_metrics_df)
display(final_predictions_df.head())

Initializing NUTS using jitter+adapt_diag...
Sequential sampling (2 chains in 1 job)
NUTS: [alpha, beta_fixed, sigma_group_intercept, z_group_intercept, f_price_per_litre_global_slope, f_price_per_litre_sigma_group_slope, f_price_per_litre_z_group_slope, nb_alpha]
c:\Users\Samuel\Desktop\fypcleanded\.venv\Lib\site-packages\pytensor\tensor\blas.py:239: RuntimeWarning: invalid value encountered in dot
  out = np.dot(A, x)
Sampling 2 chains for 300 tune and 300 draw iterations (600 + 600 draws total) took 15173 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details
c:\Users\Samuel\Desktop\fypcleanded\.venv\Lib\

,Model,Model Objective,Tuning Method,Tuning Objective,Refit Method,column_list,MAE,RMSE,RMSLE,R2,...,Refit Time (s),End-to-End Time (s),Inference Latency (ms/row),Model Size (bytes),Posterior Draws,Divergences,Max Rhat,Min ESS Bulk,Model Params (json),Monotone Constraints Config (json)
0,Bayesian Hierarchical Regression,negative_binomial_log_link,Optuna TPE,mean_rmse,NUTS,"[""price_per_litre"", ""customer"", ""promotion_ind...",4413.112285,11968.04931,1.851259,0.569219,...,15191.51462,16844.794167,0.059429,286489951.0,600,0,NaN,NaN,"{""model_config"": {""model_objective"": ""negative...",{}


,product_sku_code,customer,year,yearweek,nielsen_total_volume,promotion_indicator,top_brand,flavor_internal,pack_type_internal,pack_size_internal,...,week,week_sin,week_cos,continuous_week,__series_group_key,y_true,y_pred,y_pred_p05,y_pred_p50,y_pred_p95
73887,201050,L2_ASDA,2025,202515,387,0,COCA-COLA,COLA,GLASS,250,...,15,0.970942,-0.239316,118,COCA-COLA,387.0,587.226679,542.233551,587.226679,635.869046
73888,201050,L2_MORRISONS,2025,202515,608,0,COCA-COLA,COLA,GLASS,250,...,15,0.970942,-0.239316,118,COCA-COLA,608.0,393.135373,364.123585,393.135373,424.801911
73889,201050,L2_SAINSBURY'S,2025,202515,1781,0,COCA-COLA,COLA,GLASS,250,...,15,0.970942,-0.239316,118,COCA-COLA,1781.0,560.744801,519.825023,560.744801,604.701327
73890,201051,L2_SAINSBURY'S,2025,202515,805,0,COCA-COLA LIGHT / DIET COKE,COLA,GLASS,250,...,15,0.970942,-0.239316,118,COCA-COLA LIGHT / DIET COKE,805.0,560.318764,512.512386,560.318764,612.519487
73891,207883,L2_ASDA,2025,202515,1230,1,APPLETISER,APPLE,CAN,150,...,15,0.970942,-0.239316,118,APPLETISER,1230.0,2073.812127,1962.159015,2073.812127,2180.614244


In [19]:
# =========================
# Stage 5 - save final results
# =========================
final_metrics_df.to_csv(TEST_METRICS_PATH, index=False)
final_predictions_df.to_csv(TEST_PREDICTIONS_PATH, index=False)
print(f"Saved: {TEST_METRICS_PATH}")
print(f"Saved: {TEST_PREDICTIONS_PATH}")


Saved: results\bayesian_hierarchical_regression_group_brand_slope_price\5_test_metrics.csv
Saved: results\bayesian_hierarchical_regression_group_brand_slope_price\5_test_predictions.csv


In [20]:
# =========================
# Stage 5 - save final model
# =========================
joblib.dump(artifact_bundle, BEST_MODEL_PATH)
az.to_netcdf(final_artifact["idata"], BEST_TRACE_PATH)
print(f"Saved: {BEST_MODEL_PATH}")
print(f"Saved: {BEST_TRACE_PATH}")


Saved: results\bayesian_hierarchical_regression_group_brand_slope_price\best_model.joblib
Saved: results\bayesian_hierarchical_regression_group_brand_slope_price\best_trace.nc
